In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/smallamp/training_dataset.csv
/kaggle/input/smallamp/test_dataset.csv
/kaggle/input/ffpnet/pytorch/default/1/best_toxic_model.pt
/kaggle/input/ffpnet/pytorch/default/1/head_only.pt
/kaggle/input/ampdata/training_dataset.csv
/kaggle/input/ampdata/test_dataset.csv
/kaggle/input/data-toxic/test_toxic.csv
/kaggle/input/data-toxic/train_toxic.csv
/kaggle/input/111111/dplm-main/train.py
/kaggle/input/111111/dplm-main/env.yml
/kaggle/input/111111/dplm-main/setup.cfg
/kaggle/input/111111/dplm-main/LICENSE
/kaggle/input/111111/dplm-main/generate_dplm.py
/kaggle/input/111111/dplm-main/test.py
/kaggle/input/111111/dplm-main/README.md
/kaggle/input/111111/dplm-main/generate_dplm2.py
/kaggle/input/111111/dplm-main/requirements.txt
/kaggle/input/111111/dplm-main/setup.py
/kaggle/input/111111/dplm-main/assets/dplm.png
/kaggle/input/111111/dplm-main/assets/co_generation.png
/kaggle/input/111111/dplm-main/assets/dplm-2.png
/kaggle/input/111111/dplm-main/scripts/download_motif_scaffolds.sh

In [4]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5EncoderModel
from torch.optim import AdamW

train = pd.read_csv("/kaggle/input/smallamp/training_dataset.csv")
test = pd.read_csv("/kaggle/input/smallamp/test_dataset.csv")

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
tokenizer = T5Tokenizer.from_pretrained('Rostlab/prot_t5_xl_half_uniref50-enc', do_lower_case=False)

class ProteinDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=100):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        seq = self.data.loc[idx, "sequence"]
        label = self.data.loc[idx, "class"]
        seq = " ".join(list(seq))

        encoding = self.tokenizer(
            seq,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(label, dtype=torch.long)
        }

train_dataset = ProteinDataset(train, tokenizer)
test_dataset = ProteinDataset(test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=200, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=200, shuffle=True)



In [5]:
class ProteinClassifier(nn.Module):
    def __init__(self, encoder_name="Rostlab/prot_t5_xl_half_uniref50-enc", num_classes=2, hidden_dim=512, nhead=4, nlayers=2):
        super().__init__()
        self.encoder = T5EncoderModel.from_pretrained(encoder_name)

        embed_dim = self.encoder.config.d_model
        
        self.conv1 = nn.Conv1d(in_channels=embed_dim, out_channels=100, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(in_channels=embed_dim, out_channels=100, kernel_size=5, padding=2)
        self.conv3 = nn.Conv1d(in_channels=embed_dim, out_channels=100, kernel_size=7, padding=3)

        encoder_layer = nn.TransformerEncoderLayer(d_model=300, nhead=nhead, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=nlayers)

        self.classifier = nn.Sequential(
            nn.Linear(300, 300 // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(300 // 2, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        with torch.no_grad(): 
            outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
            hidden_states = outputs.last_hidden_state  # [batch, seq_len, embed_dim]

        # CNN expects [batch, embed_dim, seq_len]
        x = hidden_states.transpose(1, 2)
       
        x1,x2,x3= torch.relu(self.conv1(x)),torch.relu(self.conv2(x)),torch.relu(self.conv3(x))
        x = torch.cat((x1,x2,x3),dim=1)
        
        x = x.transpose(1, 2)  # [batch, seq_len, hidden_dim]

        x = self.transformer(x)

        pooled = x.mean(dim=1)

        return self.classifier(pooled)

In [6]:
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, f1_score
def _update_confmat(preds, labels):
    """
    preds, labels: torch tensors of shape [B], values in {0,1}
    returns: tp, tn, fp, fn (ints)
    """
    preds = preds.detach().cpu()
    labels = labels.detach().cpu()

    tp = ((preds == 1) & (labels == 1)).sum().item()
    tn = ((preds == 0) & (labels == 0)).sum().item()
    fp = ((preds == 1) & (labels == 0)).sum().item()
    fn = ((preds == 0) & (labels == 1)).sum().item()

    return tp, tn, fp, fn

def _sens_spec(tp, tn, fp, fn, eps=1e-12):
    sensitivity = tp / (tp + fn + eps)   # Recall for positive class
    specificity = tn / (tn + fp + eps)   # Recall for negative class
    return sensitivity, specificity
    
def train_model(model, train_loader, test_loader, epochs=5, lr=1e-4, save_path="best_model.pt"):
    model.to(device)
    optimizer = AdamW(model.parameters(), lr=lr)
    class_weights = torch.tensor([1.0, 50], device=device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    best_loss = 1e9
    
    for epoch in range(epochs):
        # -------------------- TRAIN --------------------
        model.train()
        total_loss, correct = 0, 0
        tp_tr = tn_tr = fp_tr = fn_tr = 0
        n_samples = 0

        for i, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            probs = torch.softmax(logits, dim=1)[:,1]
            preds = (probs > 0.5).long()
            correct += (preds == labels).sum().item()
            n_samples += labels.size(0)

            tp, tn, fp, fn = _update_confmat(preds, labels)
            tp_tr += tp; tn_tr += tn; fp_tr += fp; fn_tr += fn

            if i >= 20:
                break

        train_acc = correct /n_samples
        train_sens, train_spec = _sens_spec(tp_tr, tn_tr, fp_tr, fn_tr)

        # -------------------- EVAL --------------------
        model.eval()
        correct, test_loss = 0, 0
        tp_te = tn_te = fp_te = fn_te = 0

        all_probs = []
        all_labels = []
        n_samples = 0
        
        for i, batch in enumerate(tqdm(test_loader, desc=f"Epoch {epoch+1}/{epochs} [Eval]", leave=False)):
            with torch.no_grad():
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                logits = model(input_ids, attention_mask)
                loss = criterion(logits, labels)
                test_loss += loss.item()

                probs = torch.softmax(logits, dim=1)[:, 1]   
                preds = (probs > 0.5).long()
                
                correct += (preds == labels).sum().item()
                n_samples += labels.size(0)
                
                tp, tn, fp, fn = _update_confmat(preds, labels)
                tp_te += tp; tn_te += tn; fp_te += fp; fn_te += fn

                all_probs.append(probs.cpu())
                all_labels.append(labels.cpu())

            if i >= 20:
                break

        all_probs = torch.cat(all_probs).numpy()
        all_labels = torch.cat(all_labels).numpy()

        test_acc = correct / n_samples
        test_sens, test_spec = _sens_spec(tp_te, tn_te, fp_te, fn_te)

        # ---------- New metrics ----------
        roc_auc = roc_auc_score(all_labels, all_probs)
        pr_auc = average_precision_score(all_labels, all_probs)
        precision = precision_score(all_labels, (all_probs > 0.5).astype(int))
        f1 = f1_score(all_labels, (all_probs > 0.5).astype(int))

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train Acc: {train_acc:.3f} | Train Sens: {train_sens:.3f} | Train Spec: {train_spec:.3f} || "
            f"Test Acc: {test_acc:.3f} | Test Sens: {test_sens:.3f} | Test Spec: {test_spec:.3f} | "
            f"ROC-AUC: {roc_auc:.4f} | PR-AUC: {pr_auc:.4f} | Precision: {precision:.4f} | F1: {f1:.4f}"
        )

        if test_loss < best_loss:
            best_loss = test_loss
            torch.save({
                "conv1": model.conv1.state_dict(),
                "conv2": model.conv2.state_dict(),
                "conv3": model.conv3.state_dict(),
                "transformer": model.transformer.state_dict(),
                "classifier": model.classifier.state_dict()
            }, "head_only.pt")
            print(f"✔️ Epoch {epoch+1}: New best model saved")

    print(f"Training finished. Best loss: {best_loss:.4f}")

model=ProteinClassifier()
train_model(model, train_loader, test_loader, epochs=3)

pytorch_model.bin:   0%|          | 0.00/2.42G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.42G [00:00<?, ?B/s]


Epoch 1/3 [Train]:   5%|▌         | 20/380 [02:16<40:48,  6.80s/it]


Epoch 1/3 | Train Acc: 0.883 | Train Sens: 0.885 | Train Spec: 0.883 || Test Acc: 0.958 | Test Sens: 0.932 | Test Spec: 0.958 | ROC-AUC: 0.9862 | PR-AUC: 0.8100 | Precision: 0.1907 | F1: 0.3166
✔️ Epoch 1: New best model saved


Epoch 2/3 | Train Acc: 0.957 | Train Sens: 0.959 | Train Spec: 0.957 || Test Acc: 0.973 | Test Sens: 0.907 | Test Spec: 0.974 | ROC-AUC: 0.9789 | PR-AUC: 0.8509 | Precision: 0.2653 | F1: 0.4105


Epoch 3/3 | Train Acc: 0.969 | Train Sens: 0.941 | Train Spec: 0.970 || Test Acc: 0.926 | Test Sens: 0.952 | Test Spec: 0.925 | ROC-AUC: 0.9781 | PR-AUC: 0.8256 | Precision: 0.1143 | F1: 0.2041
Training finished. Best loss: 2.9691


In [ ]:
amp_model = ProteinClassifier(num_classes=2)

checkpoint = torch.load("/kaggle/input/ffpnet/pytorch/default/1/head_only.pt", map_location=device)
amp_model.conv1.load_state_dict(checkpoint["conv1"])
amp_model.conv2.load_state_dict(checkpoint["conv2"])
amp_model.conv3.load_state_dict(checkpoint["conv3"])
amp_model.transformer.load_state_dict(checkpoint["transformer"])
amp_model.classifier.load_state_dict(checkpoint["classifier"])
amp_model.to(device)
amp_model.eval()

In [ ]:
seqs = ["MSAAKEIAATEEKRLKSSDKLSATEAAWKOVPSKEVIAEEOTALRNOVEPFVKWLKGKTEPGRESLEDTMGILKALIRIR"]
batch = [" ".join(list(s)) for s in seqs]
batch = [s.replace("U","X").replace("Z","X").replace("O","X") for s in batch]

inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True)
input_ids = inputs["input_ids"].to(device)
attention_mask = inputs["attention_mask"].to(device)

with torch.no_grad():
    logits = amp_model(input_ids, attention_mask)
    probs = torch.softmax(logits, dim=-1)
    preds = torch.argmax(probs, dim=-1).cpu().numpy()

print("Predictions:", probs) 


In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5EncoderModel
from torch.optim import AdamW
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, f1_score

train_toxic = pd.read_csv("/kaggle/input/data-toxic/train_toxic.csv")
test_toxic = pd.read_csv("/kaggle/input/data-toxic/test_toxic.csv")

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
tokenizer = T5Tokenizer.from_pretrained('Rostlab/prot_t5_xl_half_uniref50-enc', do_lower_case=False)

class ProteinDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=100):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        seq = self.data.loc[idx, "sequence"]
        label = self.data.loc[idx, "class"]
        seq = " ".join(list(seq))

        encoding = self.tokenizer(
            seq,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(label, dtype=torch.long)
        }

train_dataset = ProteinDataset(train_toxic, tokenizer)
test_dataset = ProteinDataset(test_toxic, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=True)

class ProteinClassifier(nn.Module):
    def __init__(self, encoder_name="Rostlab/prot_t5_xl_half_uniref50-enc", num_classes=2, hidden_dim=512, nhead=4, nlayers=2):
        super().__init__()
        self.encoder = T5EncoderModel.from_pretrained(encoder_name)

        embed_dim = self.encoder.config.d_model
        
        self.conv1 = nn.Conv1d(in_channels=embed_dim, out_channels=100, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(in_channels=embed_dim, out_channels=100, kernel_size=5, padding=2)
        self.conv3 = nn.Conv1d(in_channels=embed_dim, out_channels=100, kernel_size=7, padding=3)

        encoder_layer = nn.TransformerEncoderLayer(d_model=300, nhead=nhead, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=nlayers)

        self.classifier = nn.Sequential(
            nn.Linear(300, 300 // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(300 // 2, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        with torch.no_grad(): 
            outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
            hidden_states = outputs.last_hidden_state  # [batch, seq_len, embed_dim]

        # CNN expects [batch, embed_dim, seq_len]
        x = hidden_states.transpose(1, 2)
       
        x1,x2,x3= torch.relu(self.conv1(x)),torch.relu(self.conv2(x)),torch.relu(self.conv3(x))
        x = torch.cat((x1,x2,x3),dim=1)
        
        x = x.transpose(1, 2)  # [batch, seq_len, hidden_dim]

        x = self.transformer(x)

        pooled = x.mean(dim=1)

        return self.classifier(pooled)

from tqdm import tqdm
def _update_confmat(preds, labels):
    # preds, labels: 1D Long tensors of 0/1
    tp = ((preds == 1) & (labels == 1)).sum().item()
    tn = ((preds == 0) & (labels == 0)).sum().item()
    fp = ((preds == 1) & (labels == 0)).sum().item()
    fn = ((preds == 0) & (labels == 1)).sum().item()
    return tp, tn, fp, fn

def _sens_spec(tp, tn, fp, fn, eps=1e-12):
    sensitivity = tp / (tp + fn + eps)  # Recall for positive
    specificity = tn / (tn + fp + eps)  # Recall for negative
    return sensitivity, specificity
    
def train_model(model, train_loader, test_loader, epochs=5, lr=1e-4, save_path="best_model.pt"):
    model.to(device)
    optimizer = AdamW(model.parameters(), lr=lr)
    class_weights = torch.tensor([1.0, 1.6], device=device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    best_loss = 1e9
    
    for epoch in range(epochs):
        # -------------------- TRAIN --------------------
        model.train()
        total_loss, correct = 0, 0
        tp_tr = tn_tr = fp_tr = fn_tr = 0
        n_samples = 0

        for i, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            probs = torch.softmax(logits, dim=1)[:,1]
            preds = (probs > 0.5).long()
            correct += (preds == labels).sum().item()
            n_samples += labels.size(0)

            tp, tn, fp, fn = _update_confmat(preds, labels)
            tp_tr += tp; tn_tr += tn; fp_tr += fp; fn_tr += fn

            if i >= 20:
                break

        train_acc = correct /n_samples
        train_sens, train_spec = _sens_spec(tp_tr, tn_tr, fp_tr, fn_tr)

        # -------------------- EVAL --------------------
        model.eval()
        correct, test_loss = 0, 0
        tp_te = tn_te = fp_te = fn_te = 0

        all_probs = []
        all_labels = []
        n_samples = 0
        
        for i, batch in enumerate(tqdm(test_loader, desc=f"Epoch {epoch+1}/{epochs} [Eval]", leave=False)):
            with torch.no_grad():
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                logits = model(input_ids, attention_mask)
                loss = criterion(logits, labels)
                test_loss += loss.item()

                probs = torch.softmax(logits, dim=1)[:, 1]   
                preds = (probs > 0.5).long()
                
                correct += (preds == labels).sum().item()
                n_samples += labels.size(0)
                
                tp, tn, fp, fn = _update_confmat(preds, labels)
                tp_te += tp; tn_te += tn; fp_te += fp; fn_te += fn

                all_probs.append(probs.cpu())
                all_labels.append(labels.cpu())

            if i >= 20:
                break

        all_probs = torch.cat(all_probs).numpy()
        all_labels = torch.cat(all_labels).numpy()

        test_acc = correct / n_samples
        test_sens, test_spec = _sens_spec(tp_te, tn_te, fp_te, fn_te)

        # ---------- New metrics ----------
        roc_auc = roc_auc_score(all_labels, all_probs)
        pr_auc = average_precision_score(all_labels, all_probs)
        precision = precision_score(all_labels, (all_probs > 0.5).astype(int))
        f1 = f1_score(all_labels, (all_probs > 0.5).astype(int))

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train Acc: {train_acc:.3f} | Train Sens: {train_sens:.3f} | Train Spec: {train_spec:.3f} || "
            f"Test Acc: {test_acc:.3f} | Test Sens: {test_sens:.3f} | Test Spec: {test_spec:.3f} | "
            f"ROC-AUC: {roc_auc:.4f} | PR-AUC: {pr_auc:.4f} | Precision: {precision:.4f} | F1: {f1:.4f}"
        )

        if test_loss < best_loss:
            best_loss = test_loss
            torch.save({
                "conv1": model.conv1.state_dict(),
                "conv2": model.conv2.state_dict(),
                "conv3": model.conv3.state_dict(),
                "transformer": model.transformer.state_dict(),
                "classifier": model.classifier.state_dict()
            }, "head_only.pt")
            print(f"✔️ Epoch {epoch+1}: New best model saved")

    print(f"Training finished. Best loss: {best_loss:.4f}")
    

model = ProteinClassifier(num_classes=2)
if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs")
    model = nn.DataParallel(model)
train_model(model, train_loader, test_loader, epochs=3)

In [ ]:

torch.cuda.empty_cache()


In [ ]:
toxic_model = ProteinClassifier(num_classes=2)

checkpoint = torch.load("/kaggle/input/ffpnet/pytorch/default/1/best_toxic_model.pt", map_location=device)
toxic_model.conv1.load_state_dict(checkpoint["conv1"])
toxic_model.conv2.load_state_dict(checkpoint["conv2"])
toxic_model.conv3.load_state_dict(checkpoint["conv3"])
toxic_model.transformer.load_state_dict(checkpoint["transformer"])
toxic_model.classifier.load_state_dict(checkpoint["classifier"])
toxic_model.to(device)
toxic_model.eval()

In [ ]:
seqs = ["MKSAIIIASLGLASVLSFGANAAINQVNAEQAQNLQSMGTVSVSQVGSAPMDMRQELAAKAEKEGASSYRIIEARTGDSWHATAELYK", "SCFEGGKDCKNDCQCCGKWSYCKCPIWGLFGCSCVIGDSMVEVRKCQ"]
batch = [" ".join(list(s)) for s in seqs]
batch = [s.replace("U","X").replace("Z","X").replace("O","X") for s in batch]

inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True)
input_ids = inputs["input_ids"].to(device)
attention_mask = inputs["attention_mask"].to(device)

with torch.no_grad():
    logits = toxic_model(input_ids, attention_mask)
    probs = torch.softmax(logits, dim=-1)
    preds = torch.argmax(probs, dim=-1).cpu().numpy()

print("Predictions:", probs) 